In [42]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [44]:
# Extracting data from what the R script produced
df = pd.read_csv("/Users/chrisgee/UCSD2526/replication_project/data/ipums_extracted_v3.csv")
df = df.set_index("STATEFIP")
df

,YEAR,SERIAL,MONTH,HWTFINL,CPSID,ASECFLAG,ASECWTH,PERNUM,WTFINL,CPSIDP,...,OCC50LY,INDLY,OCC90LY,IND90LY,FULLPART,FIRMSIZE,INCTOT,INCWAGE,EDATT,EDATTLY
STATEFIP,,,,,,,,,,,,,,,,,,,,,
6,1988,1,1,NaN,1.987100e+13,NaN,NaN,1,2901.0600,19871000000101,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1988,1,1,NaN,1.987100e+13,NaN,NaN,2,2877.9600,19871000000102,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1988,1,1,NaN,1.987100e+13,NaN,NaN,3,3152.0400,19871000000103,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
34,1988,2,1,NaN,1.987100e+13,NaN,NaN,1,1241.0400,19871000000301,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,1988,3,1,NaN,1.986100e+13,NaN,NaN,1,1922.3700,19861000000501,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,1997,60517,12,299.0424,1.997091e+13,NaN,NaN,2,299.0424,19970906014702,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
56,1997,60517,12,299.0424,1.997091e+13,NaN,NaN,3,231.2252,19970906014703,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
56,1997,60517,12,299.0424,1.997091e+13,NaN,NaN,4,210.2731,19970906014704,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [45]:
# Aggregating to find data points relevant to New Jersey and Pennsylvania
nj_pa_df = df.loc[[34, 42]]

# Creating identifier for states: 1 for NJ, 0 for PA
nj_pa_df = nj_pa_df.assign(NJ_PA = lambda x: x.index == 34) 

# Creating identifier for post policy period: 1 for after 04 1992, 0 for before 04 1992 (inclusive)
nj_pa_df['POLICY_PERIOD'] = nj_pa_df.apply(lambda x: (x['YEAR'] > 1992) or (x['YEAR'] == 1992 and x['MONTH'] > 4), axis = 1)
# making indicators 1 and 0 
nj_pa_df['NJ_PA'] = nj_pa_df['NJ_PA'].astype(int)
nj_pa_df['POLICY_PERIOD'] = nj_pa_df['POLICY_PERIOD'].astype(int)
nj_pa_df = nj_pa_df.reset_index()

# Creating treatment indicator that will be used for difference in differences estimation (i.e. 1 for NJ and after 04 1992, 0 otherwise)
nj_pa_df['DID_DUMMY'] = nj_pa_df['POLICY_PERIOD'] * nj_pa_df['NJ_PA']

In [46]:
# Creating variable for formatted period that will be used for graph axis
nj_pa_df['DAY'] = 1
nj_pa_df['PERIOD'] = pd.to_datetime(nj_pa_df[['MONTH', 'YEAR', 'DAY']]).dt.to_period('M')
nj_pa_df = nj_pa_df.drop(columns=['DAY', 'STATEFIP'])
nj_pa_df = nj_pa_df[['PERIOD', 'YEAR', 'MONTH', 'IND1990', 'EMPSTAT', 'ASECWT', 'WTFINL', 'DID_DUMMY', 'NJ_PA', 'POLICY_PERIOD' ]]
nj_pa_df.to_csv('/Users/chrisgee/UCSD2526/replication_project/data/state_by_month_cps.csv')